In [20]:
#导包
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
#                             准确率           精确率            召回率        F1值        roc曲线          分类评估报告
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split


In [18]:
def dm01_data_preprocess(): #数据预处理
    #1. 读取数据
    churn_df=pd.read_csv('./data/churn.csv')
    #churn_df.info()#原始数据
    #Churn和gender都是字符串 所以需要进行独热编码
    #如果性别有三种 直接设置成0 1 2 模型会以为女性+女性=保密 所以要设置成三个独立的个体 推荐删除一列
    #2. 独热编码
    churn_df=pd.get_dummies(churn_df,columns=['Churn','gender']) #独热编码 one-hot
    #Churn->Churn_No,Churn_Yes  gender->gender_Female,gender_Male
    #churn_df.info() #修改后
    #3. 删除冗余列
    churn_df.drop(['Churn_No','gender_Male'],axis='columns',inplace=True) # 删除多余列
    churn_df.info()
    #4. 重命名
    churn_df.rename(columns={'Churn_Yes':'flag'},inplace=True) #把流失当作标签 True->流失 False->不流失
    print(churn_df.groupby('flag').size()) #查看这里面流失和不流失各有多少

dm01_data_preprocess()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Partner_att        7043 non-null   int64  
 1   Dependents_att     7043 non-null   int64  
 2   landline           7043 non-null   int64  
 3   internet_att       7043 non-null   int64  
 4   internet_other     7043 non-null   int64  
 5   StreamingTV        7043 non-null   int64  
 6   StreamingMovies    7043 non-null   int64  
 7   Contract_Month     7043 non-null   int64  
 8   Contract_1YR       7043 non-null   int64  
 9   PaymentBank        7043 non-null   int64  
 10  PaymentCreditcard  7043 non-null   int64  
 11  PaymentElectronic  7043 non-null   int64  
 12  MonthlyCharges     7043 non-null   float64
 13  TotalCharges       7043 non-null   float64
 14  Churn_Yes          7043 non-null   bool   
 15  gender_Female      7043 non-null   bool   
dtypes: bool(2), float64(2), 

In [ ]:
# 2. 定义函数, 用于显示: 月度会员的流失情况.
def dm02_watch(): #会员流失可视化情况
    # 1. 读取数据.
    data = pd.read_csv('./data/churn.csv')
    # 2. 对上述的数据做 热编码处理.
    data = pd.get_dummies(data)
    # 3. 删除列, 因为热编码之后, 会多出一个列, 我们删除掉.
    data.drop(['gender_Male', 'Churn_No'], axis=1, inplace=True)
    # 4. 修改列名.
    data.rename(columns={'Churn_Yes':'flag'}, inplace=True)
    # 5. 查看数据集的分布情况.
    print(data.flag.value_counts())
    print(data.columns) # 查看所有列名.

    # 6. 通过计数柱状图, 绘制(月度)会员的流失情况.
    # 参数x意思是: x轴的列名(是否是月度会员, 0 -> 不是会员, 1 -> 是会员)
    # 参数hue意思是: 根据hue的值, 将数据进行分类(False -> 不流失, True -> 流失)
    sns.countplot(data, x='Contract_Month', hue='flag')
    plt.show()

In [21]:
# 3. 定义函数, 用于实现: 逻辑回归模型的训练和评估.
def dm03_trainpre():#逻辑回归模型训练评估
    # 一、 加载数据.
    data = pd.read_csv('./data/churn.csv')
    #二、数据预处理
    # 2. 对上述的数据做 热编码处理.
    data = pd.get_dummies(data)
    # 3. 删除列, 因为热编码之后, 会多出一个列, 我们删除掉.
    data.drop(['gender_Male', 'Churn_No'], axis=1, inplace=True)
    # 4. 修改列名.
    data.rename(columns={'Churn_Yes':'flag'}, inplace=True)
    # 5. 查看数据集, 从中筛除: 特征列 和 标签列.
    # print(data.head(10))    # 特征列: Contract_Month, PaymentElectronic, internet_other
    # print(data.columns)     # 标签列: flag

    # 6. 拆分训练集和测试集.
    x = data[['Contract_Month', 'PaymentElectronic', 'internet_other']]
    y = data['flag']
    # print(len(x), len(y))
    # print(x.head(10))
    # print(y.head(10))
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=22)

    #三、特征工程（特征提取 特征预处理：归一化、标准化）

    #四、模型训练
    #4.1 创建模型对象
    estimator=LogisticRegression()
    #4.2 模型训练
    estimator.fit(x_train, y_train)
    #五、模型预测
    y_pred = estimator.predict(x_test)

    #六、模型评估
    print(accuracy_score(y_test, y_pred))   #accuracy_score  准确率
    print(precision_score(y_test, y_pred))  #precision_score 精确率  预测对的为正例占预测正例的百分比 真正例/真正例+伪正例
    print(recall_score(y_test, y_pred))     #recall_score    召回率  实际的正例 有多少被找出来       真正例/真正例+真反例
    print(f1_score(y_test, y_pred))         #f1_score        f1值
    print(classification_report(y_test, y_pred))
    # 包含 准确率、精确率 召回率 f1值 macro avg, weighted avg
    # macro avg     宏平均 不考虑样本权重 用于数据均衡的情况
    # weighted avg 样本权重平均 考虑样本权重求平均 用于数据不均衡
dm03_trainpre()

0.7615330021291696
0.6177606177606177
0.40302267002518893
0.4878048780487805
              precision    recall  f1-score   support

       False       0.79      0.90      0.84      1012
        True       0.62      0.40      0.49       397

    accuracy                           0.76      1409
   macro avg       0.71      0.65      0.67      1409
weighted avg       0.74      0.76      0.74      1409

